In [ ]:
#INSTALL PACKAGES
#pip install scikit-learn torch numpy pandas ollama numpy nltk

In [1]:
#CALL MODEL

#choose model
model_name = "llama3.1:8b-instruct-q8_0"

#pull mode
!ollama pull {model_name}

!ollama list

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest 
pulling cc04e85e1f86: 100% ▕██████████████████▏ 8.5 GB                         
pulling 948af2743fc7: 100% ▕██████████████████▏ 1.5 KB                         
pulling 0ba8f0e314b4: 100% ▕██████████████████▏  12 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 4a4a958ae550: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 
NAME                                                  ID              SIZE      MODIFIED               
llama3.1:8b-instruct-q8_0                             b158ded76fa0    8.5 GB    Less than a second ago    
llama3.1:8b                                           46e0c10c039e    4.

In [2]:
#IMPORT PACKAGES AND DATASETS

import pandas as pd
import numpy as np
import json
import ollama
from pathlib import Path
import nltk
from nltk.tokenize import word_tokenize
import math

#load articles for inductive coding
df = pd.read_csv('../100_relevant_articles.csv')

In [ ]:
#GENERATE INITIAL CODES

SYSTEM_PROMPT = """You're a communication researcher who is studying the news coverage of the Mpox epidemic. 
             You will be performing inductive thematic analysis. Your job to generate initial codes for articles. 
             Your expertise is crucial in identifying issue-specific frames that can sway pubic opinions and distort public discourse. 
             Avoid generic or overly broad categories. Do not assume predefined categories, it is imperative that the codes are driven by the data.
             Think carefully to determine codes that are representative of the articles but also have maximum variability between them. 

             Format your response has to be a JSON array of objects, and include nothing else in the response. Each object should contain the following keys:
             { 
                1. "Index": "A number indicating the code order",
                2. "Name": "A short name for the code (max 3 words)",
                3. "Description": "A 3-line explanation of the code",
                4. "Example": "A representative quote (max 2 sentences)". 
             }"""

USER_PROMPT = """Attached below is an article about Mpox. Identify upto 3 unique and prevelant codes from the text. Ensure that your output format adheres to the instructions provided."""

output_list = []

#sample
#df1 = df.sample(n=5, random_state=42).reset_index(drop=True)

#or annotate on the entire set
df1 = df.copy()

for i in range(len(df1)):
    #find codes for this chunk
    messages = [
            {"role": "system", 
             "content": SYSTEM_PROMPT
            },

            {
             "role": "user", 
             "content": USER_PROMPT + df1['text'][i] 
            }

                ]

    outputs = ollama.chat(model= model_name, messages= messages)
    
    #append response
    output_list.append(outputs.message.content)
    
    #track progress
    if(i%10 == 0): print(str(i) + " iterations finished")

In [ ]:
#PARSE INITIAL CODES
df_code = pd.DataFrame(columns = ["Index", "Name", "Description", "Example"])

for i in range(len(output_list)):
    codes = output_list[i]

    try:
        df_temp = pd.DataFrame(json.loads(codes), columns = ["Index", "Name", "Description", "Example"])
        df_code = pd.concat([df_code, df_temp], axis = 0) 
    
    except Exception as e: 
        continue

df_code = df_code.drop_duplicates(subset="Name", keep="first").reset_index(drop=True) #only keep unique codes
df_code['Index'] = range(len(df_code))
df_code.to_csv('llama_initial_codes.csv') #save the initial codes

In [ ]:
#CLUSTER CODES FURTHER

SYSTEM_PROMPT = """You're a communication researcher who is conducting an issue-specific narrative framing analysis on the news coverage of the Mpox epidemic. 
    A researcher has already gone over news articles and come up with a list of initial codes through inductive coding. Your job is go over a set of initial codes and cluster them to flesh out unique frames.
    In his seminal work, Entman said "Framing essentially involves selection and salience. To frame is to select some aspects of aperceived reality and make them more salient in a communicating text, in such a way as to promote a 
    particular problem definition, causal interpretation, moral evaluation, and/or treatment recommendation for the item described." Follow Entman's principles and define frames based on the 4 unique framing elements. 
    Prioritize frame clarity and uniqueness. 

    Format your response as an array of upto 3 JSON objects, and include nothing else in the response. Each object should represent a frame as follows:
    { 
        "Name": "A short name for the frame (max 3 words)",
        "Problem definition": "How do this frame define the problem at hand?",
        "Causal attribution": "What actors or forces does this frame attribute the cause of the problem to?",
        "Moral evaluation": "What value judgements are being made by this frame?"
        "Treatment recommendation": "What remedies does the frame suggest for tackling the problem?"
        "Example": "A representative quotes (max 2 lines each)". 
    }"""

USER_PROMPT = """Ensure that you adhere to the output format provided.
    Go over the attached codes and group them into frames for a framing analysis. 
    The following text contains all the identified codes as a JSON array of objects. 
    """ 

df_code = pd.read_csv('llama_initial_codes.csv')

num_chunks = 5 #process the codes in chunks since there are over 200 of them
cap = math.ceil(len(df_code)/num_chunks) #number of codes per chunk

df_frames = pd.DataFrame(columns = ["Name", "Problem definition", "Causal attribution", "Moral evaluation", "Treatment recommendation", "Example"])  #create dataframe

flist = []

for n in range(num_chunks):
    #turn the codes into a chunk
    df_temp = df_code.loc[range(n*cap, min(len(df_code), (n+1)*cap + 1)),:]
    json_obj = df_temp.to_dict(orient="records")
    codes = json.dumps(json_obj, indent=2, ensure_ascii=False)

    messages = [
                {"role": "system", 
                "content": SYSTEM_PROMPT
                },

                {
                "role": "user", 
                "content": USER_PROMPT + codes
                }

                    ]

    outputs = ollama.chat(model= model_name, messages= messages)
    flist.append(outputs.message.content)

    try:
        frames = json.loads(outputs.message.content)
        df_frames = pd.concat([df_frames, pd.DataFrame(frames)], ignore_index=True)

    except Exception as e:
        continue

    print(str(n) + " chunks processed")

#FORMAT AND SAVE OBTAINED FRAMES
df_frames.to_csv('llama_themes.csv')

1 chunks processed
3 chunks processed
4 chunks processed


In [ ]:
#REVISE FRAMES
SYSTEM_PROMPT = """You're a communication researcher who is conducting an issue-specific narrative framing analysis on the news coverage of the Mpox epidemic. 
    In his seminal work, Entman said "Framing essentially involves selection and salience. To frame is to select some aspects of aperceived reality and make them more salient in a communicating text, in such a way as to promote a 
    particular problem definition, causal interpretation, moral evaluation, and/or treatment recommendation for the item described." 
    A researcher has already gone over news articles and come up with a list of frames with Entman's framing elements. 
    Your job is to go over all the frames, group them based on conceptual similarity and produce a revised set of frames with upto 10 unique frames. 
    Follow Entman's principles and define frames based on the 4 unique framing elements. 
    Take your time to think carefully and prioritize frame clarity and uniqueness. 

    Format your response as an array of upto 10 JSON objects, and include nothing else in the response. Each object should represent a frame as follows:
    { 
        "Name": "A short name for the frame (max 3 words)",
        "Description": "2-3 lines of detailed description of the frame",
        "Problem definition": "How do this frame define the problem at hand?",
        "Causal attribution": "What actors or forces does this frame attribute the cause of the problem to?",
        "Moral evaluation": "What value judgements are being made by this frame?"
        "Treatment recommendation": "What remedies does the frame suggest for tackling the problem?"
        "Example": "A representative quotes (max 2 lines each)". 
    }"""

USER_PROMPT = """Go over the attached set of frames and produce a revised set of upto 10 unique frames for a codebook based annotation procedure. 
    The following text contains all the identified frames as a JSON array of objects. 
    """ 

#load data
df_frames = pd.read_csv('llama_themes.csv')
df_frames = df_frames.iloc[:,1:]
json_obj = df_frames.to_dict(orient="records")
frames = json.dumps(json_obj, indent=2, ensure_ascii=False)

messages = [
                {"role": "system", 
                "content": SYSTEM_PROMPT
                },

                {
                "role": "user", 
                "content": USER_PROMPT + frames
                }

            ]

outputs = ollama.chat(model= model_name, messages= messages)

frames_revised = json.loads(outputs.message.content)
df_frames_revised = pd.DataFrame(frames_revised)
df_frames_revised.to_csv('llama_frames.csv', index=False)

In [3]:
#ANNOTATE FRAMES USING LLM GENERATED CODEBOOK

df = pd.read_csv('llama_frames.csv')
llm_frames = [f.lower() for f in df['Name']]

#function to convert csv to json 
import csv
import re

def csv_to_json_codebook(csv_filename):
    """
    Convert CSV file to JSON codebook format matching the framing codebook structure
    """
    frames = []
    
    with open(csv_filename, 'r', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile)
        
        for i, row in enumerate(reader, 1):
            # Clean up any extra whitespace in keys and values
            cleaned_row = {key.strip(): value.strip() if isinstance(value, str) else value 
                          for key, value in row.items()}
            
            frame = {
                "frame": chr(64 + i),  # Sequential frame letter (A, B, C, etc.)
                "name": cleaned_row.get("Name", ""),
                "problem_definition": cleaned_row.get("Problem definition", ""),
                "causal_interpretation": cleaned_row.get("Causal attribution", ""),
                "moral_evaluation": cleaned_row.get("Moral evaluation", ""),
                "treatment_recommendation": cleaned_row.get("Treatment recommendation", ""),
                "examples": [cleaned_row.get("Example", "")] if cleaned_row.get("Example", "") else [],
            }
            
            frames.append(frame)

        frames = json.dumps(frames, indent=2, ensure_ascii=False)
        
    return frames

#convert
if __name__ == "__main__":
    # Convert the CSV to JSON
    codebook_str = csv_to_json_codebook('llama_frames.csv')

#create the output format
frame_checklist = {}
for f in llm_frames:
    frame_checklist[f.lower()] = "yes/no"

output_format = json.dumps(frame_checklist, ensure_ascii=False)


SYSTEM_PROMPT = """You're a communication researcher who is studying the news reporting of Mpox. You’ll perform a codebook assisted framing analysis on news articles. 
Identify the frames from the codebook that are present in the article. Here is the codebook with frame definitions: """ + codebook_str


USER_PROMPT =  """Identify frames in the article with the following guidelines:
1. Read the entire article carefully before coding
2. Identify all frames present in each article
3. Some articles maybe irrelevant to the Mpox epidemic or may not contain any of the frames mentioned in the codebook. In this case, mark "no" for every frame. 
4. Ensure at least 2 framing dimensions are explicitly present (unless noted otherwise)
5. Use frame descriptions and examples to guide decision
6. All coding should be based on EXPLICIT presence of the frame. Thus, if a frame isn't explicitly present, but implicitly implied, than mark "no" for that frame. 

Provide your response in a JSON array format, as follows, and include nothing else in the response: """ + output_format + """
If you are uncertain about the annotations for any frame, force a decision to choose "yes" or "no". """

#Annotate the test set now

##parsing responses from model
def parse_json_with_fallback(content_str, frame_names):
    #Strip whitespace
    content_str = content_str.strip()
    # If the string is supposed to end with '}', but doesn't, add it.
    if not content_str.endswith('}'):
        content_str += '}'

    # Now try parsing
    try:
        #if the response can be parsed
        dict_val = json.loads(content_str)
        #remove any leading or trailing white spaces in the keys
        cleaned_dict = {k.strip(): v for k, v in dict_val.items()}

        annotations = []

        #parse response for each frame
        for f in frame_names:
            try:
                f_val = str(cleaned_dict.get(f))

                binary_val = pd.NA
                if re.search("yes",f_val.strip().lower()):
                    binary_val = 1
                if re.search("no",f_val.strip().lower()):
                    binary_val = 0

            except:
                binary_val = pd.NA
            
            annotations.append(binary_val)

        return annotations
    
    except json.JSONDecodeError:
        return [pd.NA]*len(frame_names)

df_test = pd.read_csv("../../Framing/LLM/unlabelled_frames_test.csv")

#sample for testing things first
#df_sample = df_test.sample(n= 5, random_state= 42).reset_index(drop=True)

#otherwise
df_sample = df_test.copy()

df1 = df_sample

predictions = list()

for i in range(len(df1)):
    try:
        messages = [
        {"role": "system", 
         "content": SYSTEM_PROMPT
        },

        {
         "role": "user", 
         "content": USER_PROMPT + df1["text"][i] 
        }

            ]
    
        outputs = ollama.chat(model= model_name, messages= messages)

        predictions.append(outputs.message.content)
    
    #if there is an error
    except Exception as e:
            predictions.append(None)

    
    if(i%10 == 0): print(str(i) + " iterations finished")

df1[llm_frames] = pd.NA

for j in range(len(predictions)):
    content_str = predictions[j].lower()
    annotation = parse_json_with_fallback(content_str, llm_frames)
    df1.loc[j, llm_frames] = annotation

#write
df1.to_csv(model_name + "_zshot_llm_codebook.csv")

0 iterations finished
10 iterations finished
20 iterations finished
30 iterations finished
40 iterations finished
50 iterations finished
60 iterations finished
70 iterations finished
80 iterations finished
90 iterations finished


In [21]:
#CROSS CHECK THESE ANNOTATIONS WITH THE ANNOTATIONS ON HUMAN GENERATED CODEBOOK
import sklearn as sk

#true predictions are the model predictions using the human codebook
df_true = pd.read_csv('../../Framing/LLM/Predicted/'+model_name+'_zshot.csv')
#true predictions are human predictions using the human codebook
#df_true = pd.read_csv('../../Framing/LLM/labelled_frames_test.csv')

#convert model predictions on model generated codebook
df_pred = df_test
df_pred['sexual stigma and transmission routes'] = (df1['vulnerable populations frame'] | df1['stigma frame']).astype(int)
df_pred['racial disparities and stigmatising name'] = df1['racial disparities frame']
df_pred['global relations'] = df1['global health cooperation frame']
df_pred['public health failure'] = df1['resource deficit frame']
df_pred['epidemic preparedness and surveillance'] = (df1['public health emergency frame']|df1['crisis response frame']|df1['government intervention frame']).astype(int)

#COMPUTE KAPPA AND ACCURACY
def compute_scores(df_pred, df_gold, column_pred, column_gold, column_match):
    """Compute evaluation metrics"""
    df_pred_reduced = df_pred[[column_match, column_pred]]
    df_gold_reduced = df_gold[[column_match, column_gold]]
    # Join these two dataframes
    df_temp = (
        df_pred_reduced
        .merge(df_gold_reduced, on=column_match, how='inner', suffixes=('_pred', '_gold'))
        .dropna(subset=[column_pred + '_pred', column_gold + '_gold'])
        .reset_index(drop=True)
    )
    # Ensure that the columns have data in the same type
    df_temp[column_pred + '_pred'] = df_temp[column_pred + '_pred'].astype(int)
    df_temp[column_gold + '_gold'] = df_temp[column_gold + '_gold'].astype(int)
    # Compute scores
    acc = round(100 * sk.metrics.accuracy_score(df_temp[column_gold + '_gold'], df_temp[column_pred + '_pred']), 3)
    k = round(sk.metrics.cohen_kappa_score(df_temp[column_gold + '_gold'], df_temp[column_pred + '_pred']), 3)
    f1 = round(sk.metrics.f1_score(df_temp[column_gold + '_gold'], df_temp[column_pred + '_pred']), 3)
    precision = round(sk.metrics.precision_score(df_temp[column_gold + '_gold'], df_temp[column_pred + '_pred']), 3)
    recall = round(sk.metrics.recall_score(df_temp[column_gold + '_gold'], df_temp[column_pred + '_pred']), 3)
    print(f"Accuracy for {column_gold} is {acc}%, and Cohen's kappa is {k}")
    return [acc, k, f1, precision, recall]

#what are the scores looking like
scores_df = pd.DataFrame(index = df_pred.columns[7:12].tolist(),
                         columns= ['Accuracy', 'Kappa', 'F1', 'Precision', 'Recall'])

for f in df_pred.columns[7:12].tolist():
    scores_df.loc[f] = compute_scores(df_gold = df_true, df_pred = df_pred, column_match= "stories_id", column_gold= f, column_pred= f)

Accuracy for sexual stigma and transmission routes is 74.0%, and Cohen's kappa is 0.484
Accuracy for racial disparities and stigmatising name is 84.848%, and Cohen's kappa is 0.457
Accuracy for global relations is 72.727%, and Cohen's kappa is 0.326
Accuracy for public health failure is 81.818%, and Cohen's kappa is 0.518
Accuracy for epidemic preparedness and surveillance is 67.0%, and Cohen's kappa is 0.201
